In [1]:
# download upwelling
import pandas as pd
import re
from pathlib import Path

def download_upwelling_erddap(dataset_id: str, out_csv: Path) -> pd.DataFrame:
    url = f"https://coastwatch.pfeg.noaa.gov/erddap/griddap/{dataset_id}.csvp?upwelling_index,upwelling_index_anomaly"
    df = pd.read_csv(url)

    # strip units from column names like "time (UTC)" -> "time"
    df.columns = [re.sub(r"\s*\(.*\)$", "", c).strip() for c in df.columns]

    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time").sort_index()

    # rename to shorter names
    df = df.rename(columns={
        "upwelling_index": "ui",
        "upwelling_index_anomaly": "ui_anom",
    })

    out_csv.parent.mkdir(parents=True, exist_ok=True)
    df[["ui", "ui_anom"]].to_csv(out_csv)
    print("Saved:", out_csv.resolve(), "| rows:", len(df))
    return df[["ui", "ui_anom"]]

# NorCal ~ 39N 125W (erdUI39mo)
download_upwelling_erddap("erdUI39mo", Path("../../1_DATA/processed/norcal/upwelling_norcal_monthly.csv"))

# MidCal ~ 36N 122W (erdUI36mo)
download_upwelling_erddap("erdUI36mo", Path("../../1_DATA/processed/midcal/upwelling_midcal_monthly.csv"))

# SoCal ~ 33N 119W (erdUI33mo)
download_upwelling_erddap("erdUI33mo", Path("../../1_DATA/processed/socal/upwelling_socal_monthly.csv"))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/norcal/upwelling_norcal_monthly.csv | rows: 961
Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/upwelling_midcal_monthly.csv | rows: 960
Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/socal/upwelling_socal_monthly.csv | rows: 961


,ui,ui_anom
time,,
1946-01-15 00:00:00+00:00,33.0,14.0
1946-02-15 00:00:00+00:00,43.0,-5.0
1946-03-15 00:00:00+00:00,109.0,-11.0
1946-04-15 00:00:00+00:00,150.0,-28.0
1946-05-15 00:00:00+00:00,273.0,-9.0
...,...,...
2025-09-15 00:00:00+00:00,120.0,-16.0
2025-10-15 00:00:00+00:00,88.0,12.0
2025-11-15 00:00:00+00:00,12.0,-10.0


In [9]:
#make that data into features, easier to work with
import pandas as pd
from pathlib import Path

REGION = "midcal"  # change: midcal / socal

proc = Path("../../1_DATA/processed") / REGION
ui_path   = proc / f"upwelling_{REGION}_monthly.csv"
kelp_path = proc / f"kelp_timeseries_{REGION}_bbox.csv"   # adjust if name differs
out_path  = proc / f"upwelling_features_{REGION}_quarterly.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)

ui = pd.read_csv(ui_path, index_col=0, parse_dates=True).sort_index()

# --- KEY FIXES ---
ui.index = pd.to_datetime(ui.index).tz_localize(None)     # drop UTC timezone if present
for c in ["ui", "ui_anom"]:
    ui[c] = pd.to_numeric(ui[c], errors="coerce")         # force numeric

kelp = pd.read_csv(kelp_path, index_col=0, parse_dates=True).sort_index()
kelp.index = pd.to_datetime(kelp.index).tz_localize(None)

kelp_times  = pd.DatetimeIndex(kelp.index)
kelp_qstart = kelp_times.to_period("Q").to_timestamp(how="start")

q = ui.resample("QS")  # quarter start

feat = pd.DataFrame({
    "ui_q_mean": q["ui"].mean(),
    "ui_q_min":  q["ui"].min(),
    "uianom_q_mean": q["ui_anom"].mean(),
    "uianom_q_min":  q["ui_anom"].min(),
})

feat["uianom_q_mean_lag1"] = feat["uianom_q_mean"].shift(1)
feat["ui_q_mean_lag1"]     = feat["ui_q_mean"].shift(1)

feat = feat.reindex(kelp_qstart)
feat.index = kelp_times

feat.to_csv(out_path)
print("Saved:", out_path.resolve())
print("Non-NaN counts:\n", feat.notna().sum())
print(feat.head(10))

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal/upwelling_features_midcal_quarterly.csv
Non-NaN counts:
 ui_q_mean             167
ui_q_min              167
uianom_q_mean         167
uianom_q_min          167
uianom_q_mean_lag1    167
ui_q_mean_lag1        167
dtype: int64
             ui_q_mean  ui_q_min  uianom_q_mean  uianom_q_min  \
1984-02-15   45.666667      13.0       3.666667         -12.0   
1984-05-15  281.333333     231.0      93.666667          74.0   
1984-08-15  162.666667     113.0       4.333333         -12.0   
1984-11-15   31.000000      -3.0       8.333333         -10.0   
1985-02-15   46.000000      -1.0       4.000000         -12.0   
1985-05-15  192.333333     150.0       4.666667         -40.0   
1985-08-15  155.333333      84.0      -3.000000         -10.0   
1985-11-15   30.333333     -11.0       7.666667         -18.0   
1986-02-15    3.000000     -15.0     -39.000000         -54.0   
1986-05-15  187.333333     158.0      -0.333333        

In [10]:
# merge with labeled table
import pandas as pd
from pathlib import Path

REGION = "midcal"  # "midcal" or "socal"

proc = Path("../../1_DATA/processed")

# labeled table (adjust if yours lives in a subfolder)
labeled_path = proc / f"{REGION}_kelp_sst_labeled.csv"
if not labeled_path.exists():
    labeled_path = proc / f"{REGION}_kelp_sst_labeled.csv"  # fallback (same)
if not labeled_path.exists():
    # common alternative naming you used earlier
    labeled_path = proc / f"{REGION}_kelp_sst_labeled.csv"

# upwelling quarterly features (you just created this)
ui_feat_path = proc / REGION / f"upwelling_features_{REGION}_quarterly.csv"

if not labeled_path.exists():
    raise FileNotFoundError(f"Missing labeled file: {labeled_path.resolve()}")
if not ui_feat_path.exists():
    raise FileNotFoundError(f"Missing upwelling features file: {ui_feat_path.resolve()}")

d = pd.read_csv(labeled_path, index_col=0, parse_dates=True).sort_index()
ui = pd.read_csv(ui_feat_path, index_col=0, parse_dates=True).sort_index()

# make both indices comparable (quarterly timestamps; drop timezone if any)
d.index = pd.to_datetime(d.index).tz_localize(None)
ui.index = pd.to_datetime(ui.index).tz_localize(None)

# merge
merged = d.join(ui, how="left")

out_path = proc / f"{REGION}_kelp_sst_ui_labeled.csv"
merged.to_csv(out_path)
print("Saved:", out_path.resolve())
print("New columns added:", [c for c in ui.columns if c in merged.columns])
print("NaN counts (upwelling cols):\n", merged[ui.columns].isna().sum().head())
print(merged.head())

Saved: /Users/tonylin/Documents/kelp_project/1_DATA/processed/midcal_kelp_sst_ui_labeled.csv
New columns added: ['ui_q_mean', 'ui_q_min', 'uianom_q_mean', 'uianom_q_min', 'uianom_q_mean_lag1', 'ui_q_mean_lag1']
NaN counts (upwelling cols):
 ui_q_mean             0
ui_q_min              0
uianom_q_mean         0
uianom_q_min          0
uianom_q_mean_lag1    0
dtype: int64
            kelp_area    kelp_smooth  coverage  coverage_frac  kelp_q_z  \
1984-05-15   114310.0  155341.000000      4731       0.972856 -0.431096   
1984-08-15   196372.0  105917.000000      4840       0.995270 -1.000000   
1984-11-15     7069.0   85294.500000      4850       0.997327 -1.015500   
1985-11-15   166688.0  116891.333333      4726       0.971828 -0.614153   
1986-02-15      918.0   96289.000000      4798       0.986634 -1.540325   

            kelp_z_1yr  collapse  suppressed  sstanom_q_mean  sstanom_q_max  \
1984-05-15         NaN         0           0       -0.563491      -0.464906   
1984-08-15       

In [13]:
# does upwelling add info beyond heat?
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

# load your merged file (edit path if needed)
df = pd.read_csv("../../1_DATA/processed/midcal/midcal_kelp_sst_ui_labeled.csv",
                 index_col=0, parse_dates=True).sort_index()

# align timestamps (same as you’ve been doing)
df.index = df.index.to_period("Q").to_timestamp(how="start")

# predictors + target
lag_heat = 4
df["heat4"] = df["sstanom_q_max"].shift(lag_heat)
df["up1"]   = df["uianom_q_mean_lag1"]        # already lagged 1 quarter
df["int"]   = df["heat4"] * df["up1"]         # interaction (heat effect depends on upwelling)

tmp = df[["suppressed","heat4","up1","int"]].dropna()
y = tmp["suppressed"].astype(int).values

# standardize predictors (important for interaction model)
X = tmp[["heat4","up1","int"]].values
X = (X - X.mean(axis=0)) / X.std(axis=0)

def block_boot_auc(model_fn, X, y, block_len=4, B=2000, seed=0):
    n = len(y)
    starts = np.arange(0, n - block_len + 1)
    n_blocks = int(np.ceil(n / block_len))
    rng = np.random.default_rng(seed)

    aucs = []
    for _ in range(B):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, s + block_len) for s in chosen])[:n]
        Xb, yb = X[idx], y[idx]
        if np.unique(yb).size < 2:
            continue
        m = model_fn()
        m.fit(Xb, yb)
        p = m.predict_proba(Xb)[:,1]
        aucs.append(roc_auc_score(yb, p))

    aucs = np.array(aucs)
    return aucs.mean(), np.quantile(aucs, [0.025, 0.975]), len(aucs)

# models to compare
def m_heat_only():
    return LogisticRegression(class_weight="balanced", max_iter=500)

def m_heat_up_int():
    return LogisticRegression(class_weight="balanced", max_iter=500)

# heat-only
X1 = X[:, [0]]  # heat4 only
auc1, ci1, n1 = block_boot_auc(m_heat_only, X1, y, block_len=4, B=2000, seed=42)

# heat + upwelling + interaction
auc2, ci2, n2 = block_boot_auc(m_heat_up_int, X, y, block_len=4, B=2000, seed=42)

print("Heat-only AUC (bootstrap):", auc1, "95% CI:", ci1, "| resamples:", n1)
print("Heat+Upwelling+Interaction AUC (bootstrap):", auc2, "95% CI:", ci2, "| resamples:", n2)
print("Improvement (AUC2 - AUC1):", auc2 - auc1)

Heat-only AUC (bootstrap): 0.7023668863677981 95% CI: [0.55480763 0.85082686] | resamples: 2000
Heat+Upwelling+Interaction AUC (bootstrap): 0.7637800258882803 95% CI: [0.65875869 0.87707496] | resamples: 2000
Improvement (AUC2 - AUC1): 0.06141313952048222


In [14]:
# --- Is the AUC improvement significant? bootstrap CI for ΔAUC ---
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def model():
    return LogisticRegression(class_weight="balanced", max_iter=500)

def block_boot_delta_auc(X_heat, X_full, y, block_len=4, B=2000, seed=0):
    n = len(y)
    starts = np.arange(0, n - block_len + 1)
    n_blocks = int(np.ceil(n / block_len))
    rng = np.random.default_rng(seed)

    deltas = []
    for _ in range(B):
        chosen = rng.choice(starts, size=n_blocks, replace=True)
        idx = np.concatenate([np.arange(s, s + block_len) for s in chosen])[:n]

        yb = y[idx]
        if np.unique(yb).size < 2:
            continue

        m1 = model(); m1.fit(X_heat[idx], yb)
        p1 = m1.predict_proba(X_heat[idx])[:,1]
        auc1 = roc_auc_score(yb, p1)

        m2 = model(); m2.fit(X_full[idx], yb)
        p2 = m2.predict_proba(X_full[idx])[:,1]
        auc2 = roc_auc_score(yb, p2)

        deltas.append(auc2 - auc1)

    deltas = np.array(deltas)
    return deltas.mean(), np.quantile(deltas, [0.025, 0.975]), len(deltas)

delta_mean, delta_ci, n_ok = block_boot_delta_auc(X1, X, y, block_len=4, B=2000, seed=42)
print("ΔAUC mean:", delta_mean, "95% CI:", delta_ci, "| resamples:", n_ok)
print("Improvement significant?", delta_ci[0] > 0)

ΔAUC mean: 0.06141313952048223 95% CI: [-0.00386567  0.21467974] | resamples: 2000
Improvement significant? False
